In [1]:
import pandas as pd

df = pd.read_csv("C:/Users/sylwi/OneDrive/Dokumenty/GitHub/project-telco-churn/data/processed/telco_clean.csv")

y = df["Churn"]
X = df.drop(columns=["Churn"])

print("X:", X.shape)
print("y:", y.shape)
print(30 * "--")
print(y.value_counts(normalize=True).round(3))

X: (7043, 19)
y: (7043,)
------------------------------------------------------------
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [2]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,         
    stratify=y_temp,        
    random_state=42,
)

for name, X_part, y_part in [
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
]:
    print(f"{name:10s} X: {X_part.shape}  y: {y_part.shape}  churn: {y_part.mean():.3f}")

train      X: (4225, 19)  y: (4225,)  churn: 0.265
validation X: (1409, 19)  y: (1409,)  churn: 0.265
test       X: (1409, 19)  y: (1409,)  churn: 0.265


In [3]:
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()

print(f"Kolumny tekstowe ({len(cat_cols)}):")
print(cat_cols)
print(30 * "--")
print(f"Kolumny liczbowe ({len(num_cols)}):")
print(num_cols)

Kolumny tekstowe (15):
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
------------------------------------------------------------
Kolumny liczbowe (4):
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

scale_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
flag_cols = ["SeniorCitizen"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), scale_cols),
        ("flag", "passthrough", flag_cols),
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)  
X_val_prep = preprocessor.transform(X_val)           

print("train:", X_train_prep.shape)
print("validation:", X_val_prep.shape)

names = preprocessor.get_feature_names_out()
print(len(names))
print(names[:4])
print(names[-5:])

train: (4225, 45)
validation: (1409, 45)
45
['cat__gender_Female' 'cat__gender_Male' 'cat__Partner_No'
 'cat__Partner_Yes']
['cat__PaymentMethod_Mailed check' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'flag__SeniorCitizen']


## Model bazowy

Musimy mieć co przebić - DummyClassifier to najgłupszy możliwy model: zawsze odpowiada „klient nie odejdzie”

Wchodzi pipleline, żeby nie było data leakage

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

dummy = Pipeline([
    ("prep", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent")),  
])

dummy.fit(X_train, y_train)         
y_val_pred = dummy.predict(X_val)    

print("accuracy :", round(accuracy_score(y_val, y_val_pred), 3))
print("recall   :", round(recall_score(y_val, y_val_pred), 3))
print("precision:", round(precision_score(y_val, y_val_pred, zero_division=0), 3))
print("F1       :", round(f1_score(y_val, y_val_pred, zero_division=0), 3))
print(30 * "--")
print("Przewidziane klasy:", pd.Series(y_val_pred).value_counts().to_dict())

accuracy : 0.735
recall   : 0.0
precision: 0.0
F1       : 0.0
------------------------------------------------------------
Przewidziane klasy: {0: 1409}


Accuracy 73,5% wygląda przyzwoicie, ale model nie zrobił nic. To po prostu odsetek klientów, którzy zostali.
Recall 0%. Z klientów, którzy odeszli, nie wyłapał ani jednego.

Tu wychodzi, dlaczego accuracy nie ma sensu dla takiego modelu. 

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

logreg = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

logreg.fit(X_train, y_train)

y_val_pred = logreg.predict(X_val)                 
y_val_proba = logreg.predict_proba(X_val)[:, 1]    

print("accuracy :", round(accuracy_score(y_val, y_val_pred), 3))
print("recall   :", round(recall_score(y_val, y_val_pred), 3))
print("precision:", round(precision_score(y_val, y_val_pred), 3))
print("F1       :", round(f1_score(y_val, y_val_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_proba), 3))
print("PR-AUC   :", round(average_precision_score(y_val, y_val_proba), 3))
print(30 * "--")
print("Macierz pomyłek (wiersze = prawda, kolumny = predykcja):")
print(confusion_matrix(y_val, y_val_pred))

accuracy : 0.803
recall   : 0.535
precision: 0.658
F1       : 0.59
ROC-AUC  : 0.836
PR-AUC   : 0.642
------------------------------------------------------------
Macierz pomyłek (wiersze = prawda, kolumny = predykcja):
[[931 104]
 [174 200]]


Sprawdzam poprawność:

naprawdę został (0): 931 (trafnie) ; 104 (fałszywy alarm)

naprawdę odszedł (1): 174 (przeoczony) ; 200 (trafnie wyłapany)

931 + 104 = 1035 klientów, którzy zostali,
174 + 200 = 374 klientów, którzy odeszli.

Razem 1035 + 374 = 1409, czyli cała validation. Odsetek odejść to 374 / 1409 = 26,5%, zgadza się z EDA.

## Podnosimy recall

Baseline wyłapuje 53,5% odchodzących. Podsieść wykrywalność, kosztem fałszywych alarmów. Takie alarmy są tańsze niż przeoczenie faktycznego odejścia

Wprowadzenie *class_weight="balanced"*

In [7]:
logreg_bal = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

logreg_bal.fit(X_train, y_train)

y_val_pred = logreg_bal.predict(X_val)
y_val_proba = logreg_bal.predict_proba(X_val)[:, 1]

print("accuracy :", round(accuracy_score(y_val, y_val_pred), 3))
print("recall   :", round(recall_score(y_val, y_val_pred), 3))
print("precision:", round(precision_score(y_val, y_val_pred), 3))
print("F1       :", round(f1_score(y_val, y_val_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_proba), 3))
print("PR-AUC   :", round(average_precision_score(y_val, y_val_proba), 3))
print(30 * "--")
print(confusion_matrix(y_val, y_val_pred))

accuracy : 0.747
recall   : 0.789
precision: 0.515
F1       : 0.623
ROC-AUC  : 0.836
PR-AUC   : 0.642
------------------------------------------------------------
[[757 278]
 [ 79 295]]


Recall	wzrósł z 0,535 na 0,789, ale fałszywe alarmy wzrosły z 104 na 278. ROC-AUC i PR-AUC bez zmian. OK

In [8]:
proba = logreg.predict_proba(X_val)[:, 1]    
rows = []
for thr in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    pred = (proba >= thr).astype(int)       
    rows.append({
        "próg": thr,
        "recall": recall_score(y_val, pred),
        "precision": precision_score(y_val, pred),
        "F1": f1_score(y_val, pred),
        "alarmy": int(pred.sum()),
        "FN (przeoczeni)": int(((pred == 0) & (y_val == 1)).sum()),
        "FP (fałszywe)": int(((pred == 1) & (y_val == 0)).sum()),
    })

threshold_table = pd.DataFrame(rows).round(3)
threshold_table

,próg,recall,precision,F1,alarmy,FN (przeoczeni),FP (fałszywe)
0,0.10,0.922,0.394,0.552,875,29,530
1,0.15,0.888,0.428,0.578,775,42,443
2,0.20,0.834,0.465,0.597,671,62,359
3,0.25,0.802,0.507,0.621,592,74,292
4,0.30,0.751,0.537,0.627,523,93,242
5,0.35,0.706,0.567,0.629,466,110,202
6,0.40,0.644,0.595,0.619,405,133,164
7,0.45,0.591,0.633,0.611,349,153,128
8,0.50,0.535,0.658,0.590,304,174,104


In [9]:
best_row = threshold_table.loc[threshold_table['F1'].idxmax()]
print("\n" + "="*50)
print(f"Najlepszy próg wg F1-score: {best_row['próg']}")
print(f"Precision: {best_row['precision']} | Recall: {best_row['recall']} | F1: {best_row['F1']}")
print(f"Liczba alarmów (wskazanych do retencji): {best_row['alarmy']}")
print(f"Przeoczeni klienci (FN): {best_row['FN (przeoczeni)']}")
print(f"Fałszywe alarmy (FP): {best_row['FP (fałszywe)']}")
print("="*50)


Najlepszy próg wg F1-score: 0.35
Precision: 0.567 | Recall: 0.706 | F1: 0.629
Liczba alarmów (wskazanych do retencji): 466.0
Przeoczeni klienci (FN): 110.0
Fałszywe alarmy (FP): 202.0


In [15]:
from sklearn.preprocessing import FunctionTransformer

protect_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]


def add_features(data):
    """Dodaje nowe cechy. Działa wierszami, więc nie 'uczy się' niczego z danych (brak data leakage)."""
    d = data.copy()
    d["n_protect"] = d[protect_cols].eq("Yes").sum(axis=1)                       # 0-4
    d["ServiceCount"] = (
        d[service_cols].eq("Yes").sum(axis=1)
        + d["InternetService"].isin(["DSL", "Fiber optic"]).astype(int)
    )                                                                              # 0-9
    d["ChargePerService"] = d["MonthlyCharges"] / (d["ServiceCount"] + 1)
    d["fiber_no_support"] = (
        (d["InternetService"] == "Fiber optic")
        & (d["OnlineSecurity"] != "Yes")
        & (d["TechSupport"] != "Yes")
    ).astype(int)
    return d


def make_model(extra_scale=(), extra_flag=()):
    """Pipeline: nowe cechy -> preprocessing -> regresja logistyczna."""
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), ["tenure", "MonthlyCharges", "TotalCharges", *extra_scale]),
        ("flag", "passthrough", ["SeniorCitizen", *extra_flag]),
    ])
    return Pipeline([
        ("features", FunctionTransformer(add_features)),
        ("prep", prep),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])


variants = {
    "baseline (bez nowych cech)": make_model(),
    "+ n_protect": make_model(extra_scale=["n_protect"]),
    "+ fiber_no_support": make_model(extra_flag=["fiber_no_support"]),
    "+ ServiceCount": make_model(extra_scale=["ServiceCount"]),
    "+ ChargePerService": make_model(extra_scale=["ChargePerService"]),
    "+ n_protect + fiber_no_support": make_model(["n_protect"], ["fiber_no_support"]),
}

THRESHOLD = 0.35
rows = []
for name, model in variants.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= THRESHOLD).astype(int)
    rows.append({
        "wariant": name,
        "PR-AUC": average_precision_score(y_val, proba),
        "ROC-AUC": roc_auc_score(y_val, proba),
        f"recall@{THRESHOLD}": recall_score(y_val, pred),
        f"precision@{THRESHOLD}": precision_score(y_val, pred),
    })

pd.DataFrame(rows).round(3)

,wariant,PR-AUC,ROC-AUC,recall@0.35,precision@0.35
0,baseline (bez nowych cech),0.642,0.836,0.706,0.567
1,+ n_protect,0.642,0.836,0.706,0.567
2,+ fiber_no_support,0.642,0.836,0.698,0.565
3,+ ServiceCount,0.642,0.836,0.706,0.567
4,+ ChargePerService,0.642,0.836,0.706,0.567
5,+ n_protect + fiber_no_support,0.641,0.835,0.698,0.565


In [16]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

new_scale = ["n_protect", "ServiceCount"]
new_flag = ["fiber_no_support"]


def make_pipe(model, use_new_features):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", StandardScaler(), ["tenure", "MonthlyCharges", "TotalCharges"] + (new_scale if use_new_features else [])),
        ("flag", "passthrough", ["SeniorCitizen"] + (new_flag if use_new_features else [])),
    ])
    return Pipeline([("features", FunctionTransformer(add_features)), ("prep", prep), ("model", model)])


models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # te same foldy dla wszystkich modeli

rows = []
for model_name, model in models.items():
    for use_new in [False, True]:
        scores = cross_validate(
            make_pipe(model, use_new), X_train, y_train,
            cv=cv, scoring={"PR-AUC": "average_precision", "ROC-AUC": "roc_auc"},
        )
        rows.append({
            "model": model_name,
            "nowe cechy": "tak" if use_new else "nie",
            "PR-AUC (średnia)": scores["test_PR-AUC"].mean(),
            "PR-AUC (odch. std.)": scores["test_PR-AUC"].std(),
            "ROC-AUC (średnia)": scores["test_ROC-AUC"].mean(),
        })

cv_results = pd.DataFrame(rows).round(3)
cv_results

,model,nowe cechy,PR-AUC (średnia),PR-AUC (odch. std.),ROC-AUC (średnia)
0,LogisticRegression,nie,0.666,0.025,0.848
1,LogisticRegression,tak,0.666,0.025,0.848
2,RandomForest,nie,0.667,0.009,0.846
3,RandomForest,tak,0.667,0.010,0.846
4,HistGradientBoosting,nie,0.641,0.012,0.832
5,HistGradientBoosting,tak,0.639,0.010,0.828
